# Práctica 05: Transformaciones y Feature Engineering - RENIEC
**Objetivo:** Crear variables temporales, categorizar oficinas y aplicar escalado.

## 1. Carga de datos limpios

In [1]:
import pandas as pd, numpy as np
df = pd.read_csv("reniec_transacciones_limpio.csv")
print(f"Filas: {len(df):,}")

Filas: 88,584


/tmp/ipykernel_8360/3371826325.py:2: DtypeWarning: Columns (4: DISTRITO) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("reniec_transacciones_limpio.csv")


## 2. Variables temporales (Periodo y Mes Numérico)

In [2]:
mes_map = {
    'ENERO': 1, 'FEBRERO': 2, 'MARZO': 3, 'ABRIL': 4, 'MAYO': 5, 'JUNIO': 6,
    'JULIO': 7, 'AGOSTO': 8, 'SETIEMBRE': 9, 'SEPTIEMBRE': 9, 'OCTUBRE': 10, 'NOVIEMBRE': 11, 'DICIEMBRE': 12
}
df['MES_NUM'] = df['MES'].map(mes_map).fillna(1).astype(int)
df['PERIODO'] = df['ANIO'].astype(str) + '-' + df['MES_NUM'].astype(str).str.zfill(2)
df[['ANIO', 'MES', 'MES_NUM', 'PERIODO']].head(3)

,ANIO,MES,MES_NUM,PERIODO
0,2023,JULIO,7,2023-07
1,2023,JULIO,7,2023-07
2,2023,AGOSTO,8,2023-08


## 3. Normalización Min-Max y Estandarización Z-Score

In [3]:
# Min-Max
t_min, t_max = df['TRANSACCIONES'].min(), df['TRANSACCIONES'].max()
df['TRANSAC_MINMAX'] = (df['TRANSACCIONES'] - t_min) / (t_max - t_min)

# Z-Score
t_mean, t_std = df['TRANSACCIONES'].mean(), df['TRANSACCIONES'].std()
df['TRANSAC_ZSCORE'] = (df['TRANSACCIONES'] - t_mean) / t_std

# Discretización en terciles
q33, q66 = df['TRANSACCIONES'].quantile([0.33, 0.66])
df['VOLUMEN_CAT'] = pd.cut(df['TRANSACCIONES'], bins=[-1, q33, q66, 1e9], labels=['Bajo', 'Medio', 'Alto'])
df[['TRANSACCIONES', 'TRANSAC_MINMAX', 'TRANSAC_ZSCORE', 'VOLUMEN_CAT']].head(4)

,TRANSACCIONES,TRANSAC_MINMAX,TRANSAC_ZSCORE,VOLUMEN_CAT
0,69,0.000341,-0.201296,Medio
1,38,0.000185,-0.213551,Medio
2,55,0.000271,-0.206830,Medio
3,103,0.000511,-0.187854,Medio


## 4. Exportar dataset transformado

In [4]:
df.to_csv("reniec_transacciones_features.csv", index=False)
print("Archivo generado: reniec_transacciones_features.csv")

Archivo generado: reniec_transacciones_features.csv
